In [ ]:
from preprocessing import *

In [ ]:
root_to_hdf5("track_shower_v.root", "track_shower_v", "data.h5")

In [ ]:
from torch.utils.data import DataLoader, random_split

In [ ]:
from dataset import *

In [ ]:
dataset = LArTPCSequenceDataset("data.h5")

In [ ]:
# Fractional split
train_frac = 0.6
n_total = len(dataset)
n_train = int(train_frac * n_total)
n_val = n_total - n_train

train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_fn_pad, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=collate_fn_pad, pin_memory=True)

In [ ]:
for batch in train_loader:
    break

In [ ]:
batch['hits'][0][:,0]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(batch['hits'][0][:,0], batch['hits'][0][:,1], s=1)

# Train

In [ ]:
device = torch.device("cpu")
#device = torch.device("cuda:0")
num_classes = 4    # mip, hip, shower, lowe
class_weights = compute_class_weights(train_loader, num_classes, device=device)

In [ ]:
class_weights

In [ ]:
from network import *
model = LArTPCTransformer(
    input_dim=11,        # [x_rel, z_rel, x_abs, z_abs, width, adc, r, cosθ, sinθ, wire_pitch, wire_angle]
    embed_dim=128, num_heads=8, ff_dim=256, num_layers=4,
    num_classes=num_classes,
    dropout=0.1)
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=-1, weight=class_weights)
num_epochs = 10

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params}")

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/lar_tpc_experiment2")
global_step = 0

In [ ]:
import torch
import os
from training import *
from tqdm.notebook import tqdm

def save_checkpoint(state, filename):
    torch.save(state, filename)


best_val_loss = float("inf")   # or track best_val_acc instead
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in tqdm(range(num_epochs), "Training"):
    train_loss, train_acc, global_step = train_one_epoch(model, train_loader, optimizer, criterion, device, writer=writer, global_step=global_step)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    writer.add_scalar("Loss/Train_epoch", train_loss, epoch)
    writer.add_scalar("Loss/Validation", val_loss, epoch)
    writer.add_scalar("Accuracy/Train", train_acc, epoch)
    writer.add_scalar("Accuracy/Validation", val_acc, epoch)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    # --- Save every epoch ---
    epoch_path = os.path.join(checkpoint_dir, f"epoch_{epoch:03d}.pt")
    save_checkpoint(checkpoint, epoch_path)

    # --- Save best model ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = os.path.join(checkpoint_dir, "best_model.pt")
        save_checkpoint(checkpoint, best_path)